# HPO Data Exploration

## Research Project

**Research question:** How does the performance of HPO semantic similarity methods differ between synthetic patients and clinical cases from the literature?

### Objectives
- Explore the Human Phenotype Ontology (HPO).
- Understand the structure of the ontology.
- Analyze disease-phenotype annotations.
- Prepare the data for semantic similarity calculations.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Define project paths
PROJECT_ROOT = Path.cwd().parent
HPO_DIR = PROJECT_ROOT / "data" / "hpo"

OBO_FILE = HPO_DIR / "hp-base.obo"
HPOA_FILE = HPO_DIR / "phenotype.hpoa"

print("Project root:", PROJECT_ROOT)
print("HPO directory:", HPO_DIR)
print("Ontology file exists:", OBO_FILE.exists())
print("Annotation file exists:", HPOA_FILE.exists())

Project root: D:\uni_ 3\ProiectCercetare\hpo-similarity-synthetic-vs-clinical
HPO directory: D:\uni_ 3\ProiectCercetare\hpo-similarity-synthetic-vs-clinical\data\hpo
Ontology file exists: True
Annotation file exists: True


In [2]:
import obonet
import networkx as nx

# Load the HPO ontology
hpo_graph = obonet.read_obo(OBO_FILE)

print("Ontology loaded successfully!")
print("Number of nodes:", hpo_graph.number_of_nodes())
print("Number of edges:", hpo_graph.number_of_edges())

Ontology loaded successfully!
Number of nodes: 19904
Number of edges: 24453


In [4]:
# Inspect a few HPO terms
sample_terms = [
    "HP:0001250",  # Seizures
    "HP:0001263",  # Global developmental delay
    "HP:0004322",  # Short stature
]

for term_id in sample_terms:
    if term_id in hpo_graph:
        term = hpo_graph.nodes[term_id]

        print(f"\nID: {term_id}")
        print("Name:", term.get("name"))
        print("Namespace:", term.get("namespace"))
        print("Parents:", list(hpo_graph.successors(term_id)))
    else:
        print(f"\nTerm {term_id} not found.")


ID: HP:0001250
Name: Seizure
Namespace: None
Parents: ['HP:0012638']

ID: HP:0001263
Name: Global developmental delay
Namespace: None
Parents: ['HP:0012758']

ID: HP:0004322
Name: Short stature
Namespace: None
Parents: ['HP:0000002', 'HP:0001510']


In [5]:
# Inspect the relationships of the three HPO terms

for term_id in sample_terms:
    print(f"\n{'=' * 50}")
    print("Term:", term_id)
    print("Name:", hpo_graph.nodes[term_id].get("name"))

    print("Successors (outgoing relationships):")
    for node in hpo_graph.successors(term_id):
        print(" ", node, "-", hpo_graph.nodes[node].get("name"))

    print("Predecessors (incoming relationships):")
    for node in hpo_graph.predecessors(term_id):
        print(" ", node, "-", hpo_graph.nodes[node].get("name"))


Term: HP:0001250
Name: Seizure
Successors (outgoing relationships):
  HP:0012638 - Abnormal nervous system physiology
Predecessors (incoming relationships):
  HP:0002069 - Bilateral tonic-clonic seizure
  HP:0002133 - Status epilepticus
  HP:0002197 - Generalized-onset seizure
  HP:0007359 - Focal-onset seizure
  HP:0011145 - Symptomatic seizures
  HP:0011146 - Dialeptic seizure
  HP:0020207 - Reflex seizure
  HP:0020219 - Motor seizure
  HP:0031951 - Nocturnal seizures
  HP:0032807 - Neonatal seizure
  HP:0032892 - Infection-related seizure
  HP:0033259 - Non-motor seizure

Term: HP:0001263
Name: Global developmental delay
Successors (outgoing relationships):
  HP:0012758 - Neurodevelopmental delay
Predecessors (incoming relationships):
  HP:0011342 - Mild global developmental delay
  HP:0011343 - Moderate global developmental delay
  HP:0011344 - Severe global developmental delay
  HP:0012736 - Profound global developmental delay

Term: HP:0004322
Name: Short stature
Successors (out

In [6]:
# Load disease-phenotype annotations

annotations = pd.read_csv(
    HPOA_FILE,
    sep="\t",
    comment="#",
    dtype=str
)

print("Annotations loaded successfully!")
print("Number of annotations:", len(annotations))

print("\nColumns:")
print(annotations.columns.tolist())

print("\nFirst 5 rows:")
display(annotations.head())

Annotations loaded successfully!
Number of annotations: 286651

Columns:
['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']

First 5 rows:


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [8]:
# Basic statistics about disease-phenotype annotations

print("Number of annotations:", len(annotations))

print(
    "Number of unique diseases:",
    annotations["database_id"].nunique()
)

print(
    "Number of unique HPO terms:",
    annotations["hpo_id"].nunique()
)

print(
    "Average annotations per disease:",
    round(
        annotations.groupby("database_id")["hpo_id"].count().mean(),
        2
    )
)

print("\nAnnotation aspects:")
display(annotations["aspect"].value_counts(dropna=False))

print("\nEvidence types:")
display(annotations["evidence"].value_counts(dropna=False).head(10))

print("\nFrequency values:")
display(annotations["frequency"].value_counts(dropna=False).head(10))

Number of annotations: 286651
Number of unique diseases: 12880
Number of unique HPO terms: 11658
Average annotations per disease: 22.26

Annotation aspects:


aspect
P      268905
I        8977
C        8559
H         131
M          77
NaN         2
Name: count, dtype: int64


Evidence types:


evidence
TAS    137011
PCS    119333
IEA     30305
NaN         2
Name: count, dtype: int64


Frequency values:


frequency
NaN           63396
HP:0040283    45524
HP:0040282    40443
HP:0040281    25791
1/1           19928
2/2            7953
HP:0040284     7035
1/2            6260
3/3            4363
1/3            4206
Name: count, dtype: int64

In [9]:
columns_to_check = ["aspect", "evidence", "frequency"]

for column in columns_to_check:
    print(f"\n{'=' * 40}")
    print(f"Column: {column}")
    print("Missing values:", annotations[column].isna().sum())
    print("Unique values:", annotations[column].nunique(dropna=False))
    print(annotations[column].value_counts(dropna=False).head(10).to_string())


Column: aspect
Missing values: 2
Unique values: 6
aspect
P      268905
I        8977
C        8559
H         131
M          77
NaN         2

Column: evidence
Missing values: 2
Unique values: 4
evidence
TAS    137011
PCS    119333
IEA     30305
NaN         2

Column: frequency
Missing values: 63396
Unique values: 2338
frequency
NaN           63396
HP:0040283    45524
HP:0040282    40443
HP:0040281    25791
1/1           19928
2/2            7953
HP:0040284     7035
1/2            6260
3/3            4363
1/3            4206


In [10]:
# Inspect annotation qualifiers

print("Qualifier values:")
print(annotations["qualifier"].value_counts(dropna=False))

# Count annotations by aspect and qualifier
print("\nAspect and qualifier combinations:")
display(
    annotations.groupby(
        ["aspect", "qualifier"],
        dropna=False
    ).size().reset_index(name="count")
)

Qualifier values:
qualifier
NaN    285918
NOT       733
Name: count, dtype: int64

Aspect and qualifier combinations:


,aspect,qualifier,count
0,C,NaN,8559
1,H,NOT,7
2,H,NaN,124
3,I,NaN,8977
4,M,NaN,77
5,P,NOT,726
6,P,NaN,268179
7,NaN,NaN,2


In [11]:
# Keep only positive phenotypic annotations

phenotype_annotations = annotations[
    (annotations["aspect"] == "P") &
    (annotations["qualifier"].isna())
].copy()

print("Positive phenotypic annotations:",
      len(phenotype_annotations))

print("Unique diseases:",
      phenotype_annotations["database_id"].nunique())

print("Unique HPO terms:",
      phenotype_annotations["hpo_id"].nunique())

print("\nFirst 5 filtered annotations:")
display(phenotype_annotations.head())

Positive phenotypic annotations: 268179
Unique diseases: 12866
Unique HPO terms: 11560

First 5 filtered annotations:


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [12]:
# Check whether annotated HPO terms exist in the ontology

ontology_terms = set(hpo_graph.nodes())
annotated_terms = set(phenotype_annotations["hpo_id"].dropna())

valid_terms = annotated_terms.intersection(ontology_terms)
missing_terms = annotated_terms.difference(ontology_terms)

print("Unique annotated HPO terms:", len(annotated_terms))
print("Terms found in ontology:", len(valid_terms))
print("Terms missing from ontology:", len(missing_terms))

if missing_terms:
    print("\nExamples of missing terms:")
    print(list(missing_terms)[:10])

Unique annotated HPO terms: 11560
Terms found in ontology: 11560
Terms missing from ontology: 0


## Semantic Similarity Analysis

The next step is to implement the Resnik semantic similarity measure.

Resnik similarity evaluates two HPO terms based on the information content (IC) of their most informative common ancestor (MICA).

The information content of a term is defined as:

IC(t) = -log(P(t))

where P(t) represents the frequency of the term and its descendants in the annotation dataset.

In [13]:
from collections import defaultdict
import math

# Map each HPO term to the diseases associated with it,
# including diseases annotated with more specific descendant terms.

diseases_per_term = defaultdict(set)

# Remove duplicate disease-term associations
unique_annotations = phenotype_annotations[
    ["database_id", "hpo_id"]
].drop_duplicates()

# Propagate each annotation towards its ancestors
for disease_id, hpo_id in unique_annotations.itertuples(index=False):
    diseases_per_term[hpo_id].add(disease_id)

    # In our graph, edges point from child to parent.
    ancestors = nx.descendants(hpo_graph, hpo_id)

    for ancestor in ancestors:
        diseases_per_term[ancestor].add(disease_id)

print("Number of terms with propagated annotations:",
      len(diseases_per_term))

Number of terms with propagated annotations: 12732


In [15]:
# Total number of diseases in the filtered dataset
total_diseases = phenotype_annotations["database_id"].nunique()

# Calculate Information Content
information_content = {}

for term_id, diseases in diseases_per_term.items():
    probability = len(diseases) / total_diseases
    information_content[term_id] = -math.log(probability)

print("Number of terms with IC:", len(information_content))

# Inspect some terms
for term_id in ["HP:0001250", "HP:0001263", "HP:0004322"]:
    print(
        term_id,
        hpo_graph.nodes[term_id].get("name"),
        "IC =", round(information_content.get(term_id, 0), 4)
    )

Number of terms with IC: 12732
HP:0001250 Seizure IC = 1.4139
HP:0001263 Global developmental delay IC = 1.4978
HP:0004322 Short stature IC = 1.7472


In [16]:
def resnik_similarity(term1, term2):
    """
    Calculate Resnik semantic similarity between two HPO terms.
    """

    if term1 not in hpo_graph or term2 not in hpo_graph:
        raise ValueError("One or both HPO terms are not in the ontology.")

    # Include each term itself and all its ancestors
    ancestors1 = nx.descendants(hpo_graph, term1) | {term1}
    ancestors2 = nx.descendants(hpo_graph, term2) | {term2}

    # Find common ancestors
    common_ancestors = ancestors1.intersection(ancestors2)

    if not common_ancestors:
        return 0.0

    # Find the most informative common ancestor
    mica = max(
        common_ancestors,
        key=lambda term: information_content.get(term, 0.0)
    )

    return information_content.get(mica, 0.0)

In [17]:
term1 = "HP:0001250"
term2 = "HP:0002069"

similarity = resnik_similarity(term1, term2)

print("Term 1:", hpo_graph.nodes[term1]["name"])
print("Term 2:", hpo_graph.nodes[term2]["name"])
print("Resnik similarity:", round(similarity, 4))

Term 1: Seizure
Term 2: Bilateral tonic-clonic seizure
Resnik similarity: 1.4139


In [18]:
# Test Resnik similarity on different term pairs

test_pairs = [
    ("HP:0001250", "HP:0001250"),
    ("HP:0001250", "HP:0002069"),
    ("HP:0001250", "HP:0001263"),
]

for term1, term2 in test_pairs:
    name1 = hpo_graph.nodes[term1]["name"]
    name2 = hpo_graph.nodes[term2]["name"]

    score = resnik_similarity(term1, term2)

    print(f"{name1} vs {name2}: {score:.4f}")

Seizure vs Seizure: 1.4139
Seizure vs Bilateral tonic-clonic seizure: 1.4139
Seizure vs Global developmental delay: 0.5213


## Conclusions – HPO Data Exploration and Resnik Similarity

### 1. HPO Data Exploration

The HPO ontology was successfully loaded and contains:
- 19,904 nodes;
- 24,453 edges.

The disease-phenotype annotation dataset contains:
- 286,651 total annotations;
- 12,880 distinct diseases;
- 11,658 distinct HPO terms.

After filtering for positive phenotypic annotations (`aspect == "P"` and missing `qualifier`), the dataset contains:
- 268,179 annotations;
- 12,866 distinct diseases;
- 11,560 distinct HPO terms.

All 11,560 annotated HPO terms were found in the loaded ontology.

### 2. Information Content

Information Content (IC) was calculated using disease annotation frequencies propagated through the HPO hierarchy.

The propagation includes diseases annotated with a term and diseases annotated with its descendants.

### 3. Resnik Semantic Similarity

The Resnik similarity measure was implemented using the Information Content of the Most Informative Common Ancestor (MICA).

The implementation was tested on three term pairs:

| Term pair | Resnik score |
|---|---:|
| Seizure – Seizure | 1.4139 |
| Seizure – Bilateral tonic-clonic seizure | 1.4139 |
| Seizure – Global developmental delay | 0.5213 |

The results illustrate how Resnik similarity depends on the Information Content of the most informative common ancestor.

### 4. Next Steps

The next stages of the project will include:
- Implementing additional semantic similarity methods.
- Generating synthetic patients under different scenarios.
- Preparing clinical cases from Phenopacket Store.
- Comparing method performance on synthetic and clinical cases.
- Evaluating results using Recall@k and Mean Reciprocal Rank (MRR).